# Ubuntu vs Cowrie vs LLM reply detector (TF-IDF + Logistic Regression)

Student notebook for a **lightweight reply-source detector** trained on already-logged shell reply text.

**What this notebook does**
- Binary: `real_ubuntu` vs `honeypot` (Cowrie + Llama + Qwen pooled)
- 3-way: `ubuntu` / `cowrie` / `llm` (Llama 3.1 8B + Qwen 2.5 7B pooled)
- Features: character TF-IDF (`char_wb` 3–5) + balanced logistic regression
- Split: `GroupShuffleSplit` by `session_id` (~70/15/15, seed=25) when `session_id` is present

**What this notebook does *not* do**
- No API keys, no Magazov re-run, no LLM fine-tuning, no live honeypot demo

The full honeypot / Ubuntu replay demo stays on your laptop. Colab only needs the small text+label table.


## How to open in Google Colab

1. Go to [https://colab.research.google.com](https://colab.research.google.com)
2. **File → Upload notebook** and choose `detector_colab.ipynb`
3. Runtime → Change runtime type → **CPU** is enough (free tier)
4. Upload **one** of these (next cell):
   - `detector_rows.csv` or `detector_rows.jsonl` from `data/analysis/detector/colab_bundle/`
   - or `detector_colab_bundle.zip` (same folder)
5. Run all cells top to bottom

Expected local test metrics to reproduce (seed=25): binary accuracy ≈ **0.758**, macro-F1 ≈ **0.750**, AUROC ≈ **0.875**; 3-way accuracy ≈ **0.694**, macro-F1 ≈ **0.682**, OvR-AUROC ≈ **0.888**.


## 1. Install packages


In [ ]:
# Free Colab CPU is fine. Install only if missing.
%pip install -q "scikit-learn>=1.4" "matplotlib>=3.8" "pandas>=2.0"
import sklearn, matplotlib, pandas as pd
print("sklearn", sklearn.__version__)
print("pandas", pd.__version__)


## 2. Upload data *or* mount Google Drive

Prefer the small **colab_bundle** (`detector_rows.csv` / `.jsonl` / `.zip`).  
You can also point at Drive if you uploaded the zip there.


In [ ]:
from pathlib import Path
import zipfile

DATA_DIR = Path("/content/detector_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# --- Option A: upload from your computer (recommended) ---
USE_UPLOAD = True
# --- Option B: Google Drive path to zip/csv/jsonl (set USE_UPLOAD=False) ---
DRIVE_PATH = "/content/drive/MyDrive/detector_colab_bundle.zip"  # change if needed

def _unpack_zip(zpath: Path) -> None:
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(DATA_DIR)
    print("Extracted", zpath, "->", DATA_DIR)

if USE_UPLOAD:
    from google.colab import files  # Colab only
    print("Choose detector_rows.csv, detector_rows.jsonl, or detector_colab_bundle.zip")
    uploaded = files.upload()  # interactive file picker
    for name, blob in uploaded.items():
        dest = DATA_DIR / Path(name).name
        dest.write_bytes(blob)
        print("Saved", dest, f"({len(blob)} bytes)")
        if dest.suffix.lower() == ".zip":
            _unpack_zip(dest)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path(DRIVE_PATH)
    assert src.exists(), f"Not found: {src}"
    if src.suffix.lower() == ".zip":
        _unpack_zip(src)
    else:
        dest = DATA_DIR / src.name
        dest.write_bytes(src.read_bytes())
        print("Copied", src, "->", dest)

print("Files in", DATA_DIR)
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name, p.stat().st_size)


## 3. Load rows (text + labels)


In [ ]:
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

ANSI_RE = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]|\x1b\].*?\x07|\x1b.")
PROMPT_RE = re.compile(r"(?:^|\n)[\w.-]+@[\w.-]+:[^\n]*[$#]\s*$")


def normalize_text(text: str) -> str:
    """Same light cleanup as scripts/06_train_detector.py (safe if already cleaned)."""
    if not text:
        return ""
    t = text.replace("\r\n", "\n").replace("\r", "\n")
    t = ANSI_RE.sub("", t)
    t = t.replace("\x00", "")
    t = PROMPT_RE.sub("", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


def load_table(data_dir: Path) -> pd.DataFrame:
    csv_p = data_dir / "detector_rows.csv"
    jsonl_p = data_dir / "detector_rows.jsonl"
    if csv_p.is_file():
        df = pd.read_csv(csv_p)
        print("Loaded CSV", csv_p, "n=", len(df))
    elif jsonl_p.is_file():
        rows = []
        with jsonl_p.open(encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
        df = pd.DataFrame(rows)
        print("Loaded JSONL", jsonl_p, "n=", len(df))
    else:
        raise FileNotFoundError(
            f"Need detector_rows.csv or detector_rows.jsonl under {data_dir}"
        )
    need = {"text", "label_binary", "label_3way"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    df = df.copy()
    df["text"] = df["text"].fillna("").astype(str).map(normalize_text)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    if "session_id" not in df.columns:
        df["session_id"] = [f"row-{i}" for i in range(len(df))]
    return df


df = load_table(DATA_DIR)
print("source_system:", dict(Counter(df.get("source_system", pd.Series(dtype=str)))))
print("label_binary:", dict(Counter(df["label_binary"])))
print("label_3way:", dict(Counter(df["label_3way"])))
df.head(3)


## 4. Helpers: group split, pipeline, metrics

Matches the local experiment: `GroupShuffleSplit` by `session_id`, char TF-IDF 3–5, balanced `LogisticRegression`, seed=25.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

SEED = 25


def group_split(texts, labels, groups, seed=SEED, test_size=0.15, val_size=0.15):
    """~70/15/15 with disjoint session_id groups (same as local script)."""
    X = np.array(texts)
    y = np.array(labels)
    g = np.array(groups)
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_val_idx, test_idx = next(gss_test.split(X, y, g))
    X_tv, y_tv, g_tv = X[train_val_idx], y[train_val_idx], g[train_val_idx]
    val_frac = val_size / (1.0 - test_size)
    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed + 1)
    train_idx_rel, val_idx_rel = next(gss_val.split(X_tv, y_tv, g_tv))
    return {
        "X_train": X_tv[train_idx_rel],
        "y_train": y_tv[train_idx_rel],
        "X_val": X_tv[val_idx_rel],
        "y_val": y_tv[val_idx_rel],
        "X_test": X[test_idx],
        "y_test": y[test_idx],
        "n_train": int(len(train_idx_rel)),
        "n_val": int(len(val_idx_rel)),
        "n_test": int(len(test_idx)),
        "sessions_test": sorted(set(g[test_idx].tolist())),
    }


def make_pipeline(seed=SEED) -> Pipeline:
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    max_features=20000,
                    lowercase=True,
                    strip_accents="unicode",
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=seed,
                    solver="lbfgs",
                ),
            ),
        ]
    )


def evaluate(pipe, X, y, label_names, binary: bool):
    pred = pipe.predict(X)
    proba = pipe.predict_proba(X)
    classes = list(pipe.named_steps["clf"].classes_)
    report = classification_report(
        y, pred, labels=label_names, output_dict=True, zero_division=0
    )
    out = {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "per_class_f1": {
            lab: float(report[lab]["f1-score"]) for lab in label_names if lab in report
        },
        "per_class_support": {
            lab: int(report[lab]["support"]) for lab in label_names if lab in report
        },
        "confusion_matrix": {
            "labels": label_names,
            "matrix": confusion_matrix(y, pred, labels=label_names).tolist(),
        },
        "y_pred": pred,
    }
    if binary and len(classes) == 2:
        preferred = ("honeypot", "llm", "cowrie")
        pos = next((p for p in preferred if p in classes), classes[1])
        pos_idx = classes.index(pos)
        out["auroc"] = float(roc_auc_score((y == pos).astype(int), proba[:, pos_idx]))
        out["positive_class"] = pos
        out["auroc_name"] = "AUROC"
    elif not binary and len(classes) >= 3:
        try:
            out["auroc"] = float(
                roc_auc_score(
                    y, proba, multi_class="ovr", average="macro", labels=classes
                )
            )
        except ValueError:
            out["auroc"] = None
        out["auroc_name"] = "AUROC (OvR macro)"
    return out


def plot_confusion(cm, labels, title):
    arr = np.array(cm)
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    im = ax.imshow(arr, cmap="Blues")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(j, i, str(arr[i, j]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    plt.show()


def print_metrics(name, m):
    print(f"=== {name} (test n={m['n']}) ===")
    print(f"accuracy : {m['accuracy']:.3f}")
    print(f"macro-F1 : {m['macro_f1']:.3f}")
    auroc = m.get("auroc")
    if auroc is not None:
        print(f"{m.get('auroc_name', 'AUROC')}: {auroc:.3f}")
    print("per-class F1:")
    for lab, f1 in m["per_class_f1"].items():
        support = m["per_class_support"][lab]
        print(f"  {lab:12s}  F1={f1:.3f}  support={support}")
    print()


print("Helpers ready. SEED =", SEED)


## 5. Train binary detector (`real_ubuntu` vs `honeypot`)


In [ ]:
label_names_bin = ["real_ubuntu", "honeypot"]
split_bin = group_split(
    df["text"].tolist(),
    df["label_binary"].tolist(),
    df["session_id"].tolist(),
    seed=SEED,
)
print(
    "split sizes:",
    split_bin["n_train"],
    split_bin["n_val"],
    split_bin["n_test"],
    "| test sessions:",
    split_bin["sessions_test"],
)

pipe_bin = make_pipeline(SEED)
pipe_bin.fit(split_bin["X_train"], split_bin["y_train"])
test_bin = evaluate(
    pipe_bin, split_bin["X_test"], split_bin["y_test"], label_names_bin, binary=True
)
print_metrics("binary", test_bin)
plot_confusion(
    test_bin["confusion_matrix"]["matrix"],
    label_names_bin,
    "binary — test confusion",
)


## 6. Train 3-way detector (`ubuntu` / `cowrie` / `llm`)


In [ ]:
label_names_3 = ["ubuntu", "cowrie", "llm"]
split_3 = group_split(
    df["text"].tolist(),
    df["label_3way"].tolist(),
    df["session_id"].tolist(),
    seed=SEED,
)
print(
    "split sizes:",
    split_3["n_train"],
    split_3["n_val"],
    split_3["n_test"],
    "| test sessions:",
    split_3["sessions_test"],
)

pipe_3 = make_pipeline(SEED)
pipe_3.fit(split_3["X_train"], split_3["y_train"])
test_3 = evaluate(
    pipe_3, split_3["X_test"], split_3["y_test"], label_names_3, binary=False
)
print_metrics("3-way", test_3)
plot_confusion(
    test_3["confusion_matrix"]["matrix"],
    label_names_3,
    "3-way — test confusion",
)


## 7. Optional pairwise checks (same as local secondary tasks)

Ubuntu vs LLM only, and Ubuntu vs Cowrie only — useful for an honest “are LLMs harder?” note, not as the primary headline.


In [ ]:
# Ubuntu vs LLM (exclude cowrie)
mask_llm = df["label_3way"].isin(["ubuntu", "llm"])
df_llm = df.loc[mask_llm].copy()
df_llm["label_ub_llm"] = np.where(
    df_llm["label_3way"] == "ubuntu", "real_ubuntu", "llm"
)
split_ul = group_split(
    df_llm["text"].tolist(),
    df_llm["label_ub_llm"].tolist(),
    df_llm["session_id"].tolist(),
    seed=SEED,
)
pipe_ul = make_pipeline(SEED)
pipe_ul.fit(split_ul["X_train"], split_ul["y_train"])
test_ul = evaluate(
    pipe_ul,
    split_ul["X_test"],
    split_ul["y_test"],
    ["real_ubuntu", "llm"],
    binary=True,
)
print_metrics("ubuntu_vs_llm", test_ul)

# Ubuntu vs Cowrie (exclude llm)
mask_cow = df["label_3way"].isin(["ubuntu", "cowrie"])
df_cow = df.loc[mask_cow].copy()
df_cow["label_ub_cow"] = np.where(
    df_cow["label_3way"] == "ubuntu", "real_ubuntu", "cowrie"
)
split_uc = group_split(
    df_cow["text"].tolist(),
    df_cow["label_ub_cow"].tolist(),
    df_cow["session_id"].tolist(),
    seed=SEED,
)
pipe_uc = make_pipeline(SEED)
pipe_uc.fit(split_uc["X_train"], split_uc["y_train"])
test_uc = evaluate(
    pipe_uc,
    split_uc["X_test"],
    split_uc["y_test"],
    ["real_ubuntu", "cowrie"],
    binary=True,
)
print_metrics("ubuntu_vs_cowrie", test_uc)


## 8. Short interpretation (honest framing)

Use these talking points for the write-up — **not** “better than all existing detectors.”

- A bag-of-character-ngrams detector **can** separate real Ubuntu stdout from honeypot replies above chance (local binary AUROC ≈ **0.88**), but it is **not** near-perfect. Treat it as a cheap auxiliary signal.
- In the 3-way fold, **Cowrie often has the highest class F1** and **pooled LLM the lowest**. That fits Cowrie looking more stereotyped once ANSI chrome is stripped, while LLM text overlaps more with natural shell output *in this feature space / fold*.
- Pairwise AUROCs (ubuntu vs LLM / ubuntu vs Cowrie) are usually close; with only ~20 Ubuntu sessions, val/test session groups are small — **do not** claim a large, stable gap.
- Prefer wording like: “exploratory TF-IDF+LR evidence that both backends remain partially detectable; class-level confusion depends on the label scheme.”
- Caveat: bootstrap CIs (see local `tables.md`) are wide. Absolute numbers are exploratory, not a large-scale claim.

If your Colab numbers match the expected seed=25 figures above (± floating noise), you reproduced the laptop experiment.


In [ ]:
# Quick comparison to expected local seed=25 test metrics
expected = {
    "binary": {"accuracy": 0.758, "macro_f1": 0.750, "auroc": 0.875},
    "3way": {"accuracy": 0.694, "macro_f1": 0.682, "auroc": 0.888},
    "ubuntu_vs_llm": {"accuracy": 0.811, "macro_f1": 0.802, "auroc": 0.927},
    "ubuntu_vs_cowrie": {"accuracy": 0.800, "macro_f1": 0.797, "auroc": 0.899},
}
got = {
    "binary": test_bin,
    "3way": test_3,
    "ubuntu_vs_llm": test_ul,
    "ubuntu_vs_cowrie": test_uc,
}
print(f"{'task':20s}  {'acc':>8}  {'F1':>8}  {'AUROC':>8}  | expected")
for k, exp in expected.items():
    m = got[k]
    print(
        f"{k:20s}  {m['accuracy']:8.3f}  {m['macro_f1']:8.3f}  "
        f"{(m.get('auroc') or float('nan')):8.3f}  | "
        f"{exp['accuracy']:.3f} / {exp['macro_f1']:.3f} / {exp['auroc']:.3f}"
    )
